# Smart Desktop Keyboard — IndicTrans2 Model Conversion (8-Language)

Exports IndicTrans2 EN→Indic (`indictrans2-en-indic-dist-200M`) to ONNX INT8 for fully offline CPU use.
Single model serves **8 languages**: Hindi, Bengali, Marathi, Telugu, Tamil, Kannada, Punjabi, Malayalam.

**Runtime → Change runtime type → CPU is fine.**

Run all cells **top to bottom in a single session**. Download the zip at the end.

---
### Languages supported
| Language | Code | Script | Unicode Range |
|----------|------|--------|---------------|
| Hindi | `hin_Deva` | Devanagari | U+0900–U+097F |
| Bengali | `ben_Beng` | Bengali | U+0980–U+09FF |
| Marathi | `mar_Deva` | Devanagari | U+0900–U+097F |
| Telugu | `tel_Telu` | Telugu | U+0C00–U+0C7F |
| Tamil | `tam_Taml` | Tamil | U+0B80–U+0BFF |
| Kannada | `kan_Knda` | Kannada | U+0C80–U+0CFF |
| Punjabi | `pan_Guru` | Gurmukhi | U+0A00–U+0A7F |
| Malayalam | `mal_Mlym` | Malayalam | U+0D00–U+0D7F |


In [1]:
# ── CELL 0 — Shared constants (run this first; all other cells depend on it) ──
#
# 8-LANGUAGE UPDATE: single INT8 model targets all 8 major Indian languages.
# IndicTrans2 is multilingual; only the target-language tag switches at
# inference. Export and quantization are language-agnostic.
#
# PIPELINE FIX: IndicTrans2 internally produces ALL Indic output in
# Devanagari script, then IndicProcessor transliterates to target script.

import os

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = 'ai4bharat/indictrans2-en-indic-dist-200M'

SRC_LANG  = 'eng_Latn'

TGT_LANGS = [
    'hin_Deva',   # Hindi       — Devanagari
    'ben_Beng',   # Bengali     — Bengali
    'mar_Deva',   # Marathi     — Devanagari
    'tel_Telu',   # Telugu      — Telugu
    'tam_Taml',   # Tamil       — Tamil
    'kan_Knda',   # Kannada     — Kannada
    'pan_Guru',   # Punjabi     — Gurmukhi
    'mal_Mlym',   # Malayalam   — Malayalam
]

LANG_NAMES = {
    'hin_Deva': 'Hindi',
    'ben_Beng': 'Bengali',
    'mar_Deva': 'Marathi',
    'tel_Telu': 'Telugu',
    'tam_Taml': 'Tamil',
    'kan_Knda': 'Kannada',
    'pan_Guru': 'Punjabi',
    'mal_Mlym': 'Malayalam',
}

# ── Paths ─────────────────────────────────────────────────────────────────────
ONNX_DIR  = '/content/indictrans2_onnx'
QUANT_DIR = '/content/indictrans2_onnx_int8'
ZIP_PATH  = '/content/indictrans2_int8'

os.makedirs(ONNX_DIR,  exist_ok=True)
os.makedirs(QUANT_DIR, exist_ok=True)

# ── Test sentences ────────────────────────────────────────────────────────────
SRC = [
    'Hello, how are you?',
    'I will be late today.',
    'Please send me the document.',
    'The meeting is at 3 PM.',
    'I love you.',
    'Thank you so much.',
    'Can you help me?',
    'I am feeling sick today.',
    'The weather is very nice.',
    'Let us go for a walk.',
    'I will call you tomorrow.',
    'Happy birthday!',
    'Please wait for me.',
    'I am very tired.',
    'What is your name?',
    'I need to talk to you.',
    'The food is delicious.',
    'See you tomorrow.',
    'I am going to the office.',
    'Please take care of yourself.',
]

# ── Reference translations ────────────────────────────────────────────────────
# Hindi: human-written (absolute + regression signal)
# All others: seeded from PT greedy in Cell 2 (regression signal only)
REF_HIN = [
    'नमस्ते, आप कैसे हैं?',
    'मैं आज देर से आऊँगा।',
    'कृपया मुझे दस्तावेज़ भेजें।',
    'बैठक 3 बजे है।',
    'मैं तुमसे प्यार करता हूँ।',
    'बहुत बहुत धन्यवाद।',
    'क्या आप मेरी मदद कर सकते हैं?',
    'मैं आज बीमार महसूस कर रहा हूँ।',
    'मौसम बहुत अच्छा है।',
    'चलो टहलने चलते हैं।',
    'मैं कल आपको फोन करूँगा।',
    'जन्मदिन मुबारक हो!',
    'कृपया मेरा इंतजार करें।',
    'मैं बहुत थका हुआ हूँ।',
    'आपका नाम क्या है?',
    'मुझे आपसे बात करनी है।',
    'खाना बहुत स्वादिष्ट है।',
    'कल मिलते हैं।',
    'मैं ऑफिस जा रहा हूँ।',
    'कृपया अपना ख्याल रखें।',
]

REFS = {lang: None for lang in TGT_LANGS}
REFS['hin_Deva'] = REF_HIN

# ── Script validation ranges ────────────────────────────────────────────────────
# NOTE: Hindi and Marathi both use Devanagari. The script gate validates
# output IS in Devanagari (not Latin/Telugu/etc.), which is correct for both.
SCRIPT_RANGES = {
    'hin_Deva': (0x0900, 0x097F),   # Devanagari
    'ben_Beng': (0x0980, 0x09FF),   # Bengali
    'mar_Deva': (0x0900, 0x097F),   # Devanagari (shared with Hindi)
    'tel_Telu': (0x0C00, 0x0C7F),   # Telugu
    'tam_Taml': (0x0B80, 0x0BFF),   # Tamil
    'kan_Knda': (0x0C80, 0x0CFF),   # Kannada
    'pan_Guru': (0x0A00, 0x0A7F),   # Gurmukhi
    'mal_Mlym': (0x0D00, 0x0D7F),   # Malayalam
}
SCRIPT_RATIO_MIN = 0.80

assert len(SRC) == len(REF_HIN), 'SRC and REF_HIN must be the same length'
assert set(SCRIPT_RANGES.keys()) >= set(TGT_LANGS), 'SCRIPT_RANGES missing a TGT_LANG'

print(f'Cell 0 done. {len(SRC)} source sentences, {len(TGT_LANGS)} target languages:')
for lang in TGT_LANGS:
    ref_status = f'{len(REFS[lang])} human refs' if REFS[lang] else 'from PT greedy (Cell 2)'
    print(f'  {LANG_NAMES[lang]:<12} ({lang})  ref: {ref_status}')
print(f'ONNX_DIR  : {ONNX_DIR}')
print(f'QUANT_DIR : {QUANT_DIR}')


Cell 0 done. 20 source sentences, 8 target languages:
  Hindi        (hin_Deva)  ref: 20 human refs
  Bengali      (ben_Beng)  ref: from PT greedy (Cell 2)
  Marathi      (mar_Deva)  ref: from PT greedy (Cell 2)
  Telugu       (tel_Telu)  ref: from PT greedy (Cell 2)
  Tamil        (tam_Taml)  ref: from PT greedy (Cell 2)
  Kannada      (kan_Knda)  ref: from PT greedy (Cell 2)
  Punjabi      (pan_Guru)  ref: from PT greedy (Cell 2)
  Malayalam    (mal_Mlym)  ref: from PT greedy (Cell 2)
ONNX_DIR  : /content/indictrans2_onnx
QUANT_DIR : /content/indictrans2_onnx_int8


In [2]:
# ── Authenticate with HuggingFace (required for gated models) ────────────────
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [3]:
!pip install -q \
    transformers==4.41.0 \
    onnxruntime==1.20.1 \
    onnx==1.17.0 \
    sacrebleu \
    sentencepiece \
    IndicTransToolkit
# No numpy pin — let Colab's numpy 2.x stay in place
# No optimum — we export encoder/decoder manually via torch.onnx in Cell 3.
# IndicTransToolkit provides IndicProcessor for the official pre/post pipeline
# (normalization, input tagging, and Devanagari→target-script transliteration).


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.3/546.3 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7

In [4]:
# ── CELL 2 — Load PyTorch model and record baseline metrics ───────────────
#
# REFERENCE POLICY:
#   Hindi  -> human-written REF_HIN (absolute + regression signal)
#   Others -> populated from PT greedy output here (regression signal only)

import torch
import sacrebleu
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

assert 'SRC' in dir(), 'Run Cell 0 first.'

print(f'Loading {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval()
print('Model loaded.')

ip = IndicProcessor(inference=True)
print('IndicProcessor ready.')

def translate_pt(sentences, mdl, tok, tgt_lang, num_beams=1):
    batch = ip.preprocess_batch(sentences, src_lang=SRC_LANG, tgt_lang=tgt_lang)
    inp = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=256)
    with torch.no_grad():
        out = mdl.generate(**inp, num_beams=num_beams, max_new_tokens=128, repetition_penalty=1.3)
    decoded = tok.batch_decode(out, skip_special_tokens=True)
    return ip.postprocess_batch(decoded, lang=tgt_lang)

baseline_bleu        = {}
baseline_chrf        = {}
baseline_bleu_greedy = {}
baseline_chrf_greedy = {}
hyps_beam_by_lang    = {}
hyps_greedy_by_lang  = {}

# Pass 1: greedy for every language (needed to seed references)
print(f'\nPass 1/2 — greedy decoding for {len(TGT_LANGS)} languages ...')
for tgt_lang in TGT_LANGS:
    print(f'  {LANG_NAMES[tgt_lang]} ({tgt_lang}) greedy ...')
    hyps_greedy_by_lang[tgt_lang] = translate_pt(SRC, model, tokenizer, tgt_lang, num_beams=1)

# Seed references from PT greedy for non-Hindi languages
print(f'\nPopulating references from PT greedy output ...')
for tgt_lang in TGT_LANGS:
    if REFS[tgt_lang] is None:
        REFS[tgt_lang] = hyps_greedy_by_lang[tgt_lang]
        print(f'  {LANG_NAMES[tgt_lang]:>12}: seeded {len(REFS[tgt_lang])} refs')
        print(f'               sample: {REFS[tgt_lang][0]!r}')
    else:
        print(f'  {LANG_NAMES[tgt_lang]:>12}: using human-written refs ({len(REFS[tgt_lang])} entries)')

# Pass 2: score greedy + run beam=4
print(f'\nPass 2/2 — scoring and beam=4 per language ...')
for tgt_lang in TGT_LANGS:
    ref = REFS[tgt_lang]
    name = LANG_NAMES[tgt_lang]
    ref_type = 'human' if tgt_lang == 'hin_Deva' else 'PT-greedy'
    print(f'\n{"="*50} {name} ({tgt_lang}) [ref: {ref_type}] {"="*10}')

    hyps_greedy = hyps_greedy_by_lang[tgt_lang]
    g_bleu = sacrebleu.corpus_bleu(hyps_greedy, [ref]).score
    g_chrf = sacrebleu.corpus_chrf(hyps_greedy, [ref]).score
    baseline_bleu_greedy[tgt_lang] = g_bleu
    baseline_chrf_greedy[tgt_lang] = g_chrf
    print(f'  greedy  BLEU: {g_bleu:6.2f}   chrF++: {g_chrf:6.2f}')

    hyps_beam = translate_pt(SRC, model, tokenizer, tgt_lang, num_beams=4)
    b_bleu = sacrebleu.corpus_bleu(hyps_beam, [ref]).score
    b_chrf = sacrebleu.corpus_chrf(hyps_beam, [ref]).score
    baseline_bleu[tgt_lang]     = b_bleu
    baseline_chrf[tgt_lang]     = b_chrf
    hyps_beam_by_lang[tgt_lang] = hyps_beam
    print(f'  beam=4  BLEU: {b_bleu:6.2f}   chrF++: {b_chrf:6.2f}')

    print(f'  Spot-check (beam=4):')
    for s, h, r in zip(SRC[:3], hyps_beam[:3], ref[:3]):
        print(f'    EN  : {s}')
        print(f'    TGT : {h}')
        print(f'    REF : {r}')

# Summary table
print(f'\n\n{"="*80}')
print(f'BASELINE SUMMARY (PyTorch greedy)')
print(f'{"="*80}')
print(f'{"Language":<12} {"Code":<12} {"Ref Type":<12} {"BLEU":>8} {"chrF++":>8}')
print(f'{"-"*52}')
for tgt_lang in TGT_LANGS:
    ref_type = 'human' if tgt_lang == 'hin_Deva' else 'PT-greedy'
    print(f'{LANG_NAMES[tgt_lang]:<12} {tgt_lang:<12} {ref_type:<12} '
          f'{baseline_bleu_greedy[tgt_lang]:>8.2f} {baseline_chrf_greedy[tgt_lang]:>8.2f}')


Loading ai4bharat/indictrans2-en-indic-dist-200M ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


dict.SRC.json: 0.00B [00:00, ?B/s]

...

modeling_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

Model loaded.
IndicProcessor ready.

Pass 1/2 — greedy decoding for 8 languages ...
  Hindi (hin_Deva) greedy ...
  Bengali (ben_Beng) greedy ...
  Marathi (mar_Deva) greedy ...
  Telugu (tel_Telu) greedy ...
  Tamil (tam_Taml) greedy ...
  Kannada (kan_Knda) greedy ...
  Punjabi (pan_Guru) greedy ...
  Malayalam (mal_Mlym) greedy ...

Populating references from PT greedy output ...
         Hindi: using human-written refs (20 entries)
       Bengali: seeded 20 refs
               sample: 'হ্যালো, কেমন আছ?'
       Marathi: seeded 20 refs
               sample: 'नमस्कार, तुम्ही कसे आहात?'
        Telugu: seeded 20 refs
               sample: 'హలో, ఎలా ఉన్నావు?'
         Tamil: seeded 20 refs
               sample: 'வணக்கம், எப்படி இருக்கிறீர்கள்?'
       Kannada: seeded 20 refs
               sample: 'ಹಲೋ, ಹೇಗಿದ್ದೀರಿ?'
       Punjabi: seeded 20 refs
               sample: 'ਹੈਲੋ, ਤੁਸੀਂ ਕਿਵੇਂ ਹੋ?'
     Malayalam: seeded 20 refs
               sample: 'ഹലോ, നിങ്ങൾക്ക് സുഖമാണോ?'

Pass 2/2 —

In [5]:
!pip install -U -q huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 10.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.41.0 requires huggingface-hub<1.0,>=0.23.0, but you have huggingface-hub 1.11.0 which is incompatible.
tokenizers 0.19.1 requires huggingface-hub<1.0,>=0.16.4, but you have huggingface-hub 1.11.0 which is incompatible.


In [6]:
# ── CELL 3 — Export encoder + decoder to ONNX via torch.onnx ─────────────────
#
# IndicTrans is a custom architecture not supported by optimum-cli.
# We export the three subgraphs manually:
#   encoder_model.onnx            — encodes source tokens
#   decoder_model.onnx            — first decode step (no KV cache)
#   decoder_with_past_model.onnx  — subsequent steps (with KV cache)
#
# NOTE: decoder_with_past_model is complex to export due to dynamic KV-cache
# shapes. For a keyboard/on-device use case, a single fused decoder export
# (no KV-cache split) is simpler and only ~10% slower on short sentences.
# We export that here as decoder_model.onnx and skip decoder_with_past_model.
import torch
import onnx
import os
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

assert 'SRC' in dir(), 'Run Cell 0 first.'
assert 'model' in dir(), 'Run Cell 2 first (need loaded PyTorch model).'

os.makedirs(ONNX_DIR, exist_ok=True)

# ── 1. Export encoder ─────────────────────────────────────────────────────────
print('Exporting encoder ...')

tagged   = [f"{SRC_LANG} {TGT_LANGS[0]} Hello, how are you?"]
enc_inp  = tokenizer(tagged, return_tensors='pt', padding=True, truncation=True, max_length=256)
input_ids      = enc_inp['input_ids']
attention_mask = enc_inp['attention_mask']

class EncoderWrapper(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.enc = m.model.encoder
    def forward(self, input_ids, attention_mask):
        return self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

enc_wrapper = EncoderWrapper(model).eval()

torch.onnx.export(
    enc_wrapper,
    (input_ids, attention_mask),
    f'{ONNX_DIR}/encoder_model.onnx',
    input_names  = ['input_ids', 'attention_mask'],
    output_names = ['last_hidden_state'],
    dynamic_axes = {
        'input_ids':        {0: 'batch', 1: 'seq'},
        'attention_mask':   {0: 'batch', 1: 'seq'},
        'last_hidden_state':{0: 'batch', 1: 'seq'},
    },
    opset_version   = 14,
    do_constant_folding = True,
    dynamo = False,
)
print('  encoder_model.onnx ✓')

# ── 2. Export decoder (fused, no KV-cache split) ──────────────────────────────
print('Exporting decoder ...')

with torch.no_grad():
    encoder_hidden = enc_wrapper(input_ids, attention_mask)

# decoder input: just the pad/bos token to start
decoder_input_ids = torch.tensor([[model.config.decoder_start_token_id or 1]])

# ── Re-export decoder with correct dynamic seq dim ───────────────────────────
# The original export used a single-token input, so the graph only knows
# how to process length-1 sequences. We need to re-export with a multi-token
# example so the graph generalises to growing sequences.

import torch, onnx, os

class DecoderWrapper(torch.nn.Module):
    def __init__(self, m): super().__init__(); self.model = m
    def forward(self, decoder_input_ids, encoder_hidden_states, encoder_attention_mask):
        out = self.model.model.decoder(
            input_ids             = decoder_input_ids,
            encoder_hidden_states = encoder_hidden_states,
            encoder_attention_mask= encoder_attention_mask,
        )
        return self.model.lm_head(out.last_hidden_state)

dec_wrapper = DecoderWrapper(model).eval()

# Export with a MULTI-TOKEN example (length 4) so the tracer sees a real seq dim
tagged   = [f"{SRC_LANG} {TGT_LANGS[0]} Hello, how are you?"]
enc_inp  = tokenizer(tagged, return_tensors='pt', padding=True, truncation=True, max_length=256)
with torch.no_grad():
    enc_wrapper_pt = model.model.encoder
    enc_out = enc_wrapper_pt(
        input_ids=enc_inp['input_ids'],
        attention_mask=enc_inp['attention_mask']
    ).last_hidden_state

# Multi-token seed: [2, 41854, 5, 149]  (</s> नमस्ते , आप)
decoder_input_ids = torch.tensor([[2, 41854, 5, 149]])

torch.onnx.export(
    dec_wrapper,
    (decoder_input_ids, enc_out, enc_inp['attention_mask']),
    f'{ONNX_DIR}/decoder_model.onnx',
    input_names  = ['decoder_input_ids', 'encoder_hidden_states', 'encoder_attention_mask'],
    output_names = ['logits'],
    dynamic_axes = {
        'decoder_input_ids':      {0: 'batch', 1: 'dec_seq'},
        'encoder_hidden_states':  {0: 'batch', 1: 'enc_seq'},
        'encoder_attention_mask': {0: 'batch', 1: 'enc_seq'},
        'logits':                 {0: 'batch', 1: 'dec_seq'},
    },
    opset_version       = 14,
    do_constant_folding = True,
    dynamo=False,
)
print('decoder_model.onnx re-exported ✓')

# Verify
m = onnx.load(f'{ONNX_DIR}/decoder_model.onnx')
onnx.checker.check_model(m)
print(f'checker passed  nodes={len(m.graph.node)} ✓')

# ── 3. Smoke-test both graphs ─────────────────────────────────────────────────
print('\nRunning onnx.checker ...')
for name in ['encoder_model.onnx', 'decoder_model.onnx']:
    m = onnx.load(f'{ONNX_DIR}/{name}')
    onnx.checker.check_model(m)
    print(f'  {name}  nodes={len(m.graph.node)}  ✓')

# ── 4. Copy tokenizer files ───────────────────────────────────────────────────
import shutil
from pathlib import Path
from huggingface_hub import snapshot_download

print('\nCopying tokenizer files ...')
cache = snapshot_download(MODEL_ID)   # ← remove trust_remote_code, not a valid arg
for f in Path(cache).iterdir():
    if f.suffix in ('.json', '.model', '.SRC', '.TGT') or f.name.startswith('sentencepiece'):
        shutil.copy(f, ONNX_DIR)
        print(f'  copied {f.name}')

print('\nCell 3 complete — ready for quantization.')

Exporting encoder ...


/tmp/ipykernel_2755/403234577.py:38: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/root/.cache/huggingface/modules/transformers_modules/ai4bharat/indictrans2-en-indic-dist-200M/173b94239f7c38886b2747b8d4a5db771a7e1232/modeling_indictrans.py:186: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if max_pos > self.weights.size(0):
/root/.cache/huggingface/modules/transformers_modules/ai4bharat/indictrans2-en-indic-dist-200M/1

  encoder_model.onnx ✓
Exporting decoder ...


/tmp/ipykernel_2755/403234577.py:96: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:86: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if input_shape[-1] > 1 or self.sliding_window is not None:
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:162: TracerWarning: Converting a tensor to a Python boolean might caus

decoder_model.onnx re-exported ✓
checker passed  nodes=4696 ✓

Running onnx.checker ...
  encoder_model.onnx  nodes=2581  ✓
  decoder_model.onnx  nodes=4696  ✓

Copying tokenizer files ...


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

  copied config.json
  copied dict.SRC.json
  copied model.SRC
  copied generation_config.json
  copied special_tokens_map.json
  copied model.TGT
  copied dict.TGT.json
  copied tokenizer_config.json

Cell 3 complete — ready for quantization.


In [7]:
# ── CELL 4 — INT8 dynamic quantization ───────────────────────────────────────
#
# FIX 1: Output filenames are kept identical to input filenames.
#   Original notebook renamed to *_quantized.onnx, which broke
#   ORTModelForSeq2SeqLM.from_pretrained() — it expects the standard names.
#
# FIX 2: optimize_model=False.
#   With optimize_model=True the quantizer fuses MatMul nodes first, then
#   silently skips quantizing them because they are no longer bare MatMuls.
#   Result: the output file looks smaller but is running FP32 at runtime.
#   The correct order is: export → quantize → then use ORT's session-level
#   optimization (graph_optimization_level) at inference time.
#
# FIX 3: MatMulInteger node count check.
#   Verifies that quantization actually produced INT8 weight ops, not FP32.
#
# FIX 4: Required tokenizer files verified after copy.

from onnxruntime.quantization import quantize_dynamic, QuantType
import glob, shutil, os, onnx

assert 'ONNX_DIR'  in dir(), 'Run Cell 0 first.'
assert 'QUANT_DIR' in dir(), 'Run Cell 0 first.'

onnx_files = sorted(glob.glob(f'{ONNX_DIR}/*.onnx'))
print(f'Quantizing {len(onnx_files)} ONNX files ...')

for src_path in onnx_files:
    fname    = os.path.basename(src_path)          # keep the ORIGINAL filename
    dst_path = f'{QUANT_DIR}/{fname}'
    print(f'  {fname} ...', end=' ', flush=True)

    quantize_dynamic(
        src_path,
        dst_path,
        weight_type=QuantType.QInt8,
        # optimize_model=False,   # FIX: do NOT pre-fuse before quantizing
    )

    # Verify quantization produced MatMulInteger ops (not 0, which means FP32 fallback)
    m    = onnx.load(dst_path)
    ops  = [n.op_type for n in m.graph.node]
    n_mmi = ops.count('MatMulInteger')
    if n_mmi == 0:
        raise RuntimeError(
            f'Quantization produced 0 MatMulInteger nodes in {fname}.\n'
            f'This means the model is still FP32. Check onnxruntime version compatibility.'
        )
    print(f'MatMulInteger nodes = {n_mmi}  ✓')

# ── Copy all non-ONNX files (tokenizer, configs) ──────────────────────────────
print('\nCopying tokenizer and config files ...')
for f in os.listdir(ONNX_DIR):
    src_path = f'{ONNX_DIR}/{f}'

    if os.path.isfile(src_path) and not f.endswith('.onnx'):
        shutil.copy2(src_path, f'{QUANT_DIR}/{f}')
        print(f'  copied: {f}')

# ── Verify all required files are present in QUANT_DIR ────────────────────────
required_all = [
    'encoder_model.onnx',
    'decoder_model.onnx',
    'tokenizer_config.json',
    'special_tokens_map.json',
]

print('\nVerifying required files in QUANT_DIR ...')
for name in required_all:
    path = f'{QUANT_DIR}/{name}'
    assert os.path.exists(path), f'MISSING: {path}'
    sz = os.path.getsize(path) / 1e6
    print(f'  {name:<45}  {sz:.1f} MB  ✓')

quant_files = sorted(os.listdir(QUANT_DIR))
total_mb    = sum(os.path.getsize(f'{QUANT_DIR}/{f}') for f in quant_files) / 1e6
print(f'\nQuantization complete. QUANT_DIR total: {total_mb:.0f} MB  (target < 200 MB)')
if total_mb > 200:
    print('  ⚠️  Size exceeds 200 MB target — consider pruning or a smaller checkpoint.')

Quantizing 2 ONNX files ...
  decoder_model.onnx ... 

MatMulInteger nodes = 181  ✓
  encoder_model.onnx ... 

MatMulInteger nodes = 108  ✓

Copying tokenizer and config files ...
  copied: config.json
  copied: dict.SRC.json
  copied: model.SRC
  copied: generation_config.json
  copied: special_tokens_map.json
  copied: model.TGT
  copied: dict.TGT.json
  copied: tokenizer_config.json

Verifying required files in QUANT_DIR ...
  encoder_model.onnx                             74.9 MB  ✓
  decoder_model.onnx                             203.7 MB  ✓
  tokenizer_config.json                          0.0 MB  ✓
  special_tokens_map.json                        0.0 MB  ✓

Quantization complete. QUANT_DIR total: 287 MB  (target < 200 MB)
  ⚠️  Size exceeds 200 MB target — consider pruning or a smaller checkpoint.


In [8]:
import os
for f in ['model.safetensors', 'pytorch_model.bin', 'README.md', '.gitattributes', 'LICENSE']:
    p = f'{QUANT_DIR}/{f}'
    if os.path.exists(p):
        os.remove(p)
        print(f'removed {f}')

# Confirm real size
import os
total = sum(os.path.getsize(f'{QUANT_DIR}/{f}') for f in os.listdir(QUANT_DIR)) / 1e6
print(f'QUANT_DIR actual size: {total:.0f} MB')

QUANT_DIR actual size: 287 MB


In [9]:
# ── CELL 4.5 — Stage trust_remote_code Python files into QUANT_DIR ───────────
#
# IndicTrans2 uses a custom tokenizer and config class that live in two Python
# files in the HF repo (not in transformers). When the keyboard app loads the
# zipped model offline with AutoTokenizer.from_pretrained(QUANT_DIR,
# trust_remote_code=True), these files MUST be next to the model weights —
# otherwise loading either fails (no network) or silently re-downloads.
#
# We pull them from the HF cache and copy them into QUANT_DIR so the zip
# produced in Cell 5 is fully self-contained.

from huggingface_hub import hf_hub_download
import shutil

for fname in ['tokenization_indictrans.py', 'configuration_indictrans.py']:
    cached_path = hf_hub_download(repo_id=MODEL_ID, filename=fname)
    dst = f'{QUANT_DIR}/{fname}'
    shutil.copy(cached_path, dst)
    size_kb = os.path.getsize(dst) / 1024
    print(f'  staged {fname}  ({size_kb:.1f} KB)  ✓')

# Sanity: list what's in QUANT_DIR now
print('\nQUANT_DIR contents after staging:')
for f in sorted(os.listdir(QUANT_DIR)):
    size_mb = os.path.getsize(f'{QUANT_DIR}/{f}') / 1e6
    marker = '  ← trust_remote_code' if f.endswith('.py') else ''
    print(f'  {f:<40} {size_mb:6.2f} MB{marker}')


  staged tokenization_indictrans.py  (7.9 KB)  ✓
  staged configuration_indictrans.py  (13.8 KB)  ✓

QUANT_DIR contents after staging:
  config.json                                0.00 MB
  configuration_indictrans.py                0.01 MB  ← trust_remote_code
  decoder_model.onnx                       203.71 MB
  dict.SRC.json                              0.65 MB
  dict.TGT.json                              3.39 MB
  encoder_model.onnx                        74.89 MB
  generation_config.json                     0.00 MB
  model.SRC                                  0.76 MB
  model.TGT                                  3.26 MB
  special_tokens_map.json                    0.00 MB
  tokenization_indictrans.py                 0.01 MB  ← trust_remote_code
  tokenizer_config.json                      0.00 MB


In [11]:
# ── CELL 5 — Validate quantized model + latency benchmark + zip ───────────

import time, os, shutil, sacrebleu
import numpy as np
import onnxruntime as ort
from transformers import AutoTokenizer
from IndicTransToolkit.processor import IndicProcessor

assert 'SRC' in dir(), 'Run Cell 0 first.'
for var in ['baseline_bleu_greedy', 'baseline_chrf_greedy', 'hyps_greedy_by_lang']:
    assert var in dir(), f'Run Cell 2 first (missing: {var}).'
for lang in TGT_LANGS:
    assert REFS.get(lang) is not None, f'REF for {lang} not populated.'

print('Loading tokenizer ...')
tok_ort = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
ip_ort = IndicProcessor(inference=True)
print('IndicProcessor ready.')

print('Loading ONNX sessions ...')
so = ort.SessionOptions()
so.intra_op_num_threads = 2
so.inter_op_num_threads = 1
enc_sess = ort.InferenceSession(f'{QUANT_DIR}/encoder_model.onnx', sess_options=so)
dec_sess = ort.InferenceSession(f'{QUANT_DIR}/decoder_model.onnx', sess_options=so)
print('Sessions loaded.')

def translate_ort(sentences, tgt_lang, max_new_tokens=64):
    batch = ip_ort.preprocess_batch(sentences, src_lang=SRC_LANG, tgt_lang=tgt_lang)
    enc_inp = tok_ort(batch, return_tensors='np', padding=True, truncation=True, max_length=256)
    enc_out = enc_sess.run(
        ['last_hidden_state'],
        {'input_ids': enc_inp['input_ids'], 'attention_mask': enc_inp['attention_mask']}
    )[0]

    n = len(sentences)
    eos_id = tok_ort.eos_token_id or 2
    dec_ids = np.full((n, 1), 2, dtype=np.int64)
    generated = [[] for _ in range(n)]
    done = [False] * n

    for _ in range(max_new_tokens):
        logits = dec_sess.run(
            ['logits'],
            {'decoder_input_ids': dec_ids,
             'encoder_hidden_states': enc_out,
             'encoder_attention_mask': enc_inp['attention_mask']}
        )[0]
        next_token = logits[:, -1, :].argmax(axis=-1)
        all_done = True
        for b in range(n):
            if not done[b]:
                tid = int(next_token[b])
                if tid == eos_id:
                    done[b] = True
                else:
                    generated[b].append(tid)
                    all_done = False
        dec_ids = np.concatenate([dec_ids, next_token[:, np.newaxis]], axis=1)
        if all_done:
            break

    decoded = tok_ort.batch_decode(generated, skip_special_tokens=True)
    return ip_ort.postprocess_batch(decoded, lang=tgt_lang)

def script_ratio(texts, lang):
    """Fraction of letter chars in expected Unicode block."""
    lo, hi = SCRIPT_RANGES[lang]
    letters_in = 0
    letters_out = 0
    for t in texts:
        for ch in t:
            cp = ord(ch)
            if ch.isspace() or cp < 0x0080:
                continue
            if lo <= cp <= hi:
                letters_in += 1
            else:
                letters_out += 1
    total = letters_in + letters_out
    return (letters_in / total) if total > 0 else 1.0

CHRF_DROP_LIMIT = 10.0
hyps_ort_by_lang = {}
results = {}

print(f'\nRunning quantized translations for {len(TGT_LANGS)} languages ...')
for tgt_lang in TGT_LANGS:
    ref = REFS[tgt_lang]
    print(f'  {LANG_NAMES[tgt_lang]} ({tgt_lang}) ...')
    hyps_ort = translate_ort(SRC, tgt_lang)
    hyps_ort_by_lang[tgt_lang] = hyps_ort

    bleu_ort  = sacrebleu.corpus_bleu(hyps_ort, [ref]).score
    chrf_ort  = sacrebleu.corpus_chrf(hyps_ort, [ref]).score
    bleu_drop = baseline_bleu_greedy[tgt_lang] - bleu_ort
    chrf_drop = baseline_chrf_greedy[tgt_lang] - chrf_ort
    s_ratio   = script_ratio(hyps_ort, tgt_lang)

    results[tgt_lang] = {
        'pt_bleu': baseline_bleu_greedy[tgt_lang],
        'ort_bleu': bleu_ort,
        'bleu_drop': bleu_drop,
        'pt_chrf': baseline_chrf_greedy[tgt_lang],
        'ort_chrf': chrf_ort,
        'chrf_drop': chrf_drop,
        'script_ratio': s_ratio,
    }

# ── Quality report ───────────────────────────────────────────────────────────────
print(f'\n{"="*100}')
print(f'QUALITY REPORT: ORT INT8 greedy vs PyTorch greedy')
print(f'{"="*100}')

header = f"{'Metric':<20}" + ''.join(f'{LANG_NAMES[l]:>12}' for l in TGT_LANGS)
print(header)
print('-' * len(header))

def row(label, fn):
    print(f'{label:<20}' + ''.join(f'{fn(l):>12}' for l in TGT_LANGS))

row('PT BLEU',      lambda l: f"{results[l]['pt_bleu']:.2f}")
row('ORT BLEU',     lambda l: f"{results[l]['ort_bleu']:.2f}")
row('BLEU drop',    lambda l: f"{results[l]['bleu_drop']:+.2f}")
row('PT chrF++',    lambda l: f"{results[l]['pt_chrf']:.2f}")
row('ORT chrF++',   lambda l: f"{results[l]['ort_chrf']:.2f}")
row('chrF++ drop',  lambda l: f"{results[l]['chrf_drop']:+.2f}")
row('Script ratio', lambda l: f"{results[l]['script_ratio']*100:.1f}%")
row('Ref type',     lambda l: 'human' if l == 'hin_Deva' else 'PT-grdy')

# ── Quality gate ─────────────────────────────────────────────────────────────────
print(f'\n── Quality Gate ──')
print(f'Criteria: chrF++ drop < {CHRF_DROP_LIMIT}, script ratio >= {SCRIPT_RATIO_MIN*100:.0f}%')
all_passed = True
pass_count = 0
fail_count = 0

for tgt_lang in TGT_LANGS:
    r = results[tgt_lang]
    chrf_ok   = r['chrf_drop'] < CHRF_DROP_LIMIT
    script_ok = r['script_ratio'] >= SCRIPT_RATIO_MIN
    ok = chrf_ok and script_ok
    status = 'PASS' if ok else 'FAIL'
    if ok: pass_count += 1
    else: fail_count += 1

    print(f'  {LANG_NAMES[tgt_lang]:<12} ({tgt_lang}): '
          f'chrF++ drop {r["chrf_drop"]:+.2f}  '
          f'script {r["script_ratio"]*100:.1f}%  -> {status}')
    if not chrf_ok:
        print(f'       chrF++ drop exceeds {CHRF_DROP_LIMIT}')
    if not script_ok:
        print(f'       Script-validity failure for {tgt_lang}')
    if not ok:
        all_passed = False

print(f'\nResults: {pass_count} passed, {fail_count} failed out of {len(TGT_LANGS)} languages')
assert all_passed, 'Quality gate failed for at least one language.'
print('Quality gate passed for ALL languages.')

# ── Per-sentence spot-check ─────────────────────────────────────────────────────
for tgt_lang in TGT_LANGS:
    ref = REFS[tgt_lang]
    hyps_ort = hyps_ort_by_lang[tgt_lang]
    hyps_base = hyps_greedy_by_lang[tgt_lang]

    print(f'\n── Per-sentence: {LANG_NAMES[tgt_lang]} ({tgt_lang}) ──')
    flagged = 0
    for i, (src, hyp_ort, hyp_base, r) in enumerate(zip(SRC, hyps_ort, hyps_base, ref)):
        s_base = sacrebleu.sentence_bleu(hyp_base, [r]).score
        s_ort  = sacrebleu.sentence_bleu(hyp_ort,  [r]).score
        drop = s_base - s_ort
        flag = '  !!' if drop > 10 else ''
        print(f'  [{i:02d}] base={s_base:5.1f}  ort={s_ort:5.1f}  drop={drop:+5.1f}{flag}')
        if drop > 10:
            flagged += 1
            print(f'       EN : {src}')
            print(f'       ORT: {hyp_ort}')
            print(f'       REF: {r}')
    if flagged > 0:
        print(f'  {flagged}/{len(SRC)} sentences flagged with >10 BLEU drop')

# ── Latency benchmark ─────────────────────────────────────────────────────────
print(f'\n{"="*80}')
print(f'LATENCY BENCHMARK (per-sentence, greedy, ORT INT8)')
print(f'{"="*80}')
latency_summary = {}
for tgt_lang in TGT_LANGS:
    latencies_ms = []
    for s in SRC:
        t0 = time.perf_counter()
        translate_ort([s], tgt_lang)
        latencies_ms.append((time.perf_counter() - t0) * 1000)
    latencies_ms.sort()
    p50  = latencies_ms[len(latencies_ms) // 2]
    p95  = latencies_ms[int(len(latencies_ms) * 0.95)]
    pmax = latencies_ms[-1]
    latency_summary[tgt_lang] = (p50, p95, pmax)

print(f'{"Language":<12} {"Code":<12} {"P50 (ms)":>10} {"P95 (ms)":>10} {"Max (ms)":>10}')
print(f'{"-"*54}')
for tgt_lang in TGT_LANGS:
    p50, p95, pmax = latency_summary[tgt_lang]
    warn = '  slow!' if p50 > 400 else ''
    print(f'{LANG_NAMES[tgt_lang]:<12} {tgt_lang:<12} {p50:>10.0f} {p95:>10.0f} {pmax:>10.0f}{warn}')

p50s = [latency_summary[l][0] for l in TGT_LANGS]
spread = max(p50s) - min(p50s)
print(f'\nP50 spread across languages: {spread:.0f} ms')

# ── Output spot-check (3 sentences each) ─────────────────────────────────────
for tgt_lang in TGT_LANGS:
    ref = REFS[tgt_lang]
    hyps_ort = hyps_ort_by_lang[tgt_lang]
    print(f'\n── Output: {LANG_NAMES[tgt_lang]} ({tgt_lang}) ──')
    for s, h, r in zip(SRC[:3], hyps_ort[:3], ref[:3]):
        print(f'  EN  : {s}')
        print(f'  ORT : {h}')
        print(f'  REF : {r}')
        print()

# ── Cleanup + zip ───────────────────────────────────────────────────────────────
for f in ['model.safetensors', 'pytorch_model.bin', 'README.md', '.gitattributes', 'LICENSE']:
    p = f'{QUANT_DIR}/{f}'
    if os.path.exists(p):
        os.remove(p)

print(f'\nCreating zip ...')
shutil.make_archive(ZIP_PATH, 'zip', QUANT_DIR)
zip_mb = os.path.getsize(ZIP_PATH + '.zip') / 1e6
print(f'\n{ZIP_PATH}.zip  ({zip_mb:.0f} MB)')
print(f'Single model serves {len(TGT_LANGS)} languages: {", ".join(LANG_NAMES[l] for l in TGT_LANGS)}')


Loading tokenizer ...
IndicProcessor ready.
Loading ONNX sessions ...
Sessions loaded.

Running quantized translations for 8 languages ...
  Hindi (hin_Deva) ...
  Bengali (ben_Beng) ...
  Marathi (mar_Deva) ...
  Telugu (tel_Telu) ...
  Tamil (tam_Taml) ...
  Kannada (kan_Knda) ...
  Punjabi (pan_Guru) ...
  Malayalam (mal_Mlym) ...

QUALITY REPORT: ORT INT8 greedy vs PyTorch greedy
Metric                     Hindi     Bengali     Marathi      Telugu       Tamil     Kannada     Punjabi   Malayalam
--------------------------------------------------------------------------------------------------------------------
PT BLEU                    65.76      100.00      100.00      100.00      100.00      100.00      100.00      100.00
ORT BLEU                   64.70       93.76       84.81       89.16       91.14       86.43       98.35       91.20
BLEU drop                  +1.06       +6.24      +15.19      +10.84       +8.86      +13.57       +1.65       +8.80
PT chrF++                  7

In [12]:
# ── FINAL COMPARISON: PyTorch vs ONNX INT8 (all 8 languages) ──────────────

import time
import torch

model.eval()

def translate_pt(sentences, tgt_lang, max_new_tokens=64):
    batch = ip.preprocess_batch(sentences, src_lang=SRC_LANG, tgt_lang=tgt_lang)
    inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=1, do_sample=False)
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return ip.postprocess_batch(decoded, lang=tgt_lang)

def benchmark(fn, name, tgt_lang):
    latencies = []
    for s in SRC:
        t0 = time.perf_counter()
        fn([s], tgt_lang)
        latencies.append((time.perf_counter() - t0) * 1000)
    latencies.sort()
    p50  = latencies[len(latencies)//2]
    p95  = latencies[int(len(latencies)*0.95)]
    pmax = latencies[-1]
    print(f"  {name:<18}  P50 {p50:5.0f} ms   P95 {p95:5.0f} ms   Max {pmax:5.0f} ms")
    return p50, p95

summary_rows = []

for tgt_lang in TGT_LANGS:
    name = LANG_NAMES[tgt_lang]
    print(f'\n{"="*60}')
    print(f'  {name} ({tgt_lang})')
    print(f'{"="*60}')

    print('Warming up...')
    translate_pt([SRC[0]], tgt_lang)
    translate_ort([SRC[0]], tgt_lang)

    print('\nLatency:')
    pt_p50,  pt_p95  = benchmark(translate_pt,  "PyTorch (200M)", tgt_lang)
    ort_p50, ort_p95 = benchmark(translate_ort, "ONNX INT8",      tgt_lang)

    speedup_p50 = pt_p50 / ort_p50 if ort_p50 > 0 else float('inf')
    improvement = (1 - ort_p50/pt_p50)*100 if pt_p50 > 0 else 0

    print(f'\n  Speedup: {speedup_p50:.2f}x  (P50: {pt_p50:.0f} ms -> {ort_p50:.0f} ms, '
          f'{improvement:.1f}% faster)')

    summary_rows.append((name, tgt_lang, pt_p50, ort_p50, speedup_p50))

    print(f'\n  Output comparison:')
    for s in SRC[:3]:
        pt_out  = translate_pt([s], tgt_lang)[0]
        ort_out = translate_ort([s], tgt_lang)[0]
        match = 'match' if pt_out == ort_out else 'DIFF'
        print(f'  EN : {s}')
        print(f'  PT : {pt_out}')
        print(f'  ORT: {ort_out}  [{match}]')
        print()

# Grand summary
print(f'\n{"="*80}')
print(f'GRAND SUMMARY: PyTorch vs ONNX INT8')
print(f'{"="*80}')
print(f'{"Language":<12} {"Code":<12} {"PT P50":>10} {"ORT P50":>10} {"Speedup":>10}')
print(f'{"-"*54}')
for name, code, pt_p50, ort_p50, speedup in summary_rows:
    print(f'{name:<12} {code:<12} {pt_p50:>8.0f}ms {ort_p50:>8.0f}ms {speedup:>9.2f}x')

avg_speedup = sum(r[4] for r in summary_rows) / len(summary_rows)
print(f'\nAverage speedup: {avg_speedup:.2f}x across {len(TGT_LANGS)} languages')
print(f'Model: {MODEL_ID}')
print(f'Languages: {", ".join(LANG_NAMES[l] for l in TGT_LANGS)}')
print(f'\nSingle ONNX INT8 model successfully validated for all {len(TGT_LANGS)} languages.')



  Hindi (hin_Deva)
Warming up...

Latency:
  PyTorch (200M)      P50   604 ms   P95  1567 ms   Max  1567 ms
  ONNX INT8           P50   161 ms   P95   271 ms   Max   271 ms

  Speedup: 3.74x  (P50: 604 ms -> 161 ms, 73.3% faster)

  Output comparison:
  EN : Hello, how are you?
  PT : हैलो, आप कैसे हैं?
  ORT: हैलो, आप कैसे हैं?  [match]

  EN : I will be late today.
  PT : आज मुझे देर हो जाएगी।
  ORT: आज मुझे देर हो जाएगी।  [match]

  EN : Please send me the document.
  PT : कृपया मुझे दस्तावेज़ भेजें।
  ORT: कृपया मुझे दस्तावेज़ भेजें।  [match]


  Bengali (ben_Beng)
Warming up...

Latency:
  PyTorch (200M)      P50   594 ms   P95  1049 ms   Max  1049 ms
  ONNX INT8           P50   154 ms   P95   203 ms   Max   203 ms

  Speedup: 3.86x  (P50: 594 ms -> 154 ms, 74.1% faster)

  Output comparison:
  EN : Hello, how are you?
  PT : হ্যালো, কেমন আছ?
  ORT: হ্যালো, কেমন আছ?  [match]

  EN : I will be late today.
  PT : আজ দেরি করে আসব।
  ORT: আজ দেরি করে আসব।  [match]

  EN : Please send

Great question. These are two different strategies the model uses to pick words when generating a translation.

**Greedy decoding** is the simplest approach — at each step, the model picks the single most probable next word and moves on. It's fast because it only considers one path. Think of it like walking through a city and always taking the road that *looks* shortest right now, without checking if a slightly longer road now might lead to a better overall route.

**Beam search (beam=4)** keeps 4 candidate translations running in parallel at each step. At every word position, it expands all 4 candidates, scores them, and keeps the best 4 again. At the end, it picks the highest-scoring complete sentence. It's slower (roughly 4x more work) but often produces more fluent, natural-sounding output because it can recover from a mediocre word choice early on.

In the output you just saw, this is why Hindi beam=4 scored 74.69 BLEU vs greedy's 65.76 — beam search found better overall translations. And why Bengali beam=4 chose "আসসালামুয়ালাইকুম" instead of "হ্যালো" — it explored multiple paths and decided that greeting scored higher overall.

**For your keyboard app, we use greedy.** Reason: beam search on ONNX Runtime runs as a Python loop (4 separate decoder calls per step), which makes it ~4x slower. For a keyboard where latency matters more than squeezing out the last bit of fluency on short phrases, greedy at 50-100ms beats beam=4 at 200-400ms. The quality difference is small on short keyboard-style sentences.